# 4장 · 조건부 확률과 베이즈 정리 🩺

웹 챕터 **"조건부 확률과 베이즈 정리"** 와 짝을 이루는 Python 실습입니다.
질병 검사 역설(양성이어도 병일 확률 9.2%)을 자연 빈도 → 공식 → 몬테카를로 시뮬레이션
세 가지 방법으로 확인합니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch4')

## 1. 1000명을 세어 보기 (자연 빈도)

유병률 1%, 민감도 90%, 건강한 990명 중 89명이 잘못 양성(위양성률 약 9%).
공식 없이 **사람 수를 그냥 셉니다**. 웹 챕터의 계산 그대로예요.

In [ ]:
total = 1000
sick = int(total * 0.01)        # 유병률 1% → 병자 10명
healthy = total - sick          # 건강한 사람 990명

true_pos = int(sick * 0.9)      # 민감도 90% → 진양성 9명
false_pos = 89                  # 건강한 990명 중 잘못 양성 (약 9%)

positives = true_pos + false_pos
print(f'병자 {sick}명 / 건강 {healthy}명')
print(f'진양성 {true_pos}명 + 위양성 {false_pos}명 = 양성 전체 {positives}명')
print(f'P(병 | 양성) = {true_pos}/{positives} = {true_pos / positives:.3f}  (약 {true_pos / positives * 100:.1f}%)')

## 2. 베이즈 공식을 함수로

$$P(\text{병} \mid \text{양성}) = \frac{P(\text{양성} \mid \text{병})\,P(\text{병})}{P(\text{양성})}$$

분모 P(양성)는 "병자가 양성" + "건강한데 양성" 두 경로의 합입니다.

In [ ]:
def posterior_prob(prevalence, sensitivity, false_rate):
    """베이즈 정리: P(병|양성). prevalence=유병률, sensitivity=민감도, false_rate=위양성률"""
    p_positive = sensitivity * prevalence + false_rate * (1 - prevalence)
    return sensitivity * prevalence / p_positive


print('본문 예시 (1%, 90%, 9%) :', round(posterior_prob(0.01, 0.90, 0.09), 3))
print('민감도를 99%로 올리면   :', round(posterior_prob(0.01, 0.99, 0.09), 3))
print('위양성률을 1%로 낮추면  :', round(posterior_prob(0.01, 0.90, 0.01), 3))

> 흥미로운 결과: 민감도를 90%→99%로 올려도 사후확률은 거의 안 오르지만(9.2%→10%),
> **위양성률을 9%→1%로 낮추면 47.6%로 뜁니다.** 희귀병 검사에서 중요한 건 위양성 줄이기예요.

## 3. 유병률이 바뀌면 사후확률은?

같은 검사라도 **사전확률(유병률)** 에 따라 양성의 의미가 완전히 달라집니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'    # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False

prev = np.linspace(0.001, 0.2, 300)
post = posterior_prob(prev, 0.9, 0.09)

plt.figure(figsize=(8, 4))
plt.plot(prev * 100, post * 100, color='#6366f1', linewidth=2)
plt.scatter([1], [posterior_prob(0.01, 0.9, 0.09) * 100], color='#f43f5e', zorder=3,
            label='본문 예시: 유병률 1% → 9.2%')
plt.xlabel('유병률 (%)')
plt.ylabel('P(병 | 양성) (%)')
plt.title('사전확률(유병률)이 사후확률을 좌우한다')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. 몬테카를로 시뮬레이션: 100만 명 검사하기

난수로 100만 명을 만들어 실제로 "양성인 사람 중 병자 비율"을 세어 봅니다.
(**몬테카를로**: 난수를 잔뜩 뽑아 확률을 근사하는 방법. 카지노 도시 이름에서 왔어요.)

In [ ]:
rng = np.random.default_rng(42)

N = 1_000_000
has_disease = rng.random(N) < 0.01               # 병 여부 (유병률 1%)
pos_if_sick = rng.random(N) < 0.90               # 병자라면 양성이 나올지
pos_if_healthy = rng.random(N) < 0.09            # 건강해도 잘못 양성이 나올지
positive = np.where(has_disease, pos_if_sick, pos_if_healthy)

sim = has_disease[positive].mean()               # 양성인 사람 중 병자 비율
print(f'양성 {positive.sum():,}명 중 병자 {(has_disease & positive).sum():,}명')
print('몬테카를로 사후확률 :', round(sim, 4))
print('공식 값             :', round(posterior_prob(0.01, 0.9, 0.09), 4))

## 5. 스팸 필터 미니 계산

메일 100통: 스팸 20통(그중 '무료' 포함 12통), 정상 80통(그중 '무료' 포함 8통).
'무료'가 든 새 메일이 스팸일 확률은?

In [ ]:
spam, ham = 20, 80
free_in_spam, free_in_ham = 12, 8

prior = spam / (spam + ham)                                # 사전확률
posterior = free_in_spam / (free_in_spam + free_in_ham)    # 사후확률
print(f'사전확률 P(스팸)          = {prior:.2f}')
print(f"사후확률 P(스팸 | '무료') = {posterior:.2f}")
print(f'단어 하나로 {prior:.0%} 에서 {posterior:.0%} 로 업데이트!')

## 정리 🎯

- **조건부 확률** P(A|B) = 좁아진 세계(B)에서 다시 센 비율. **P(A|B) ≠ P(B|A)** 방향 주의!
- **베이즈 정리** = 사전확률 → (증거) → 사후확률의 업데이트 규칙. 공식이 헷갈리면 **1000명을 세면** 됩니다.
- 질병 검사 역설: 유병률(사전확률)이 낮으면 위양성이 진양성을 압도해 **양성이어도 병일 확률이 낮다**.
- AI 연결: 이 원리를 단어 여러 개로 확장한 것이 **나이브 베이즈 분류기**(초창기 스팸 필터),
  그리고 "데이터(증거)로 믿음(모델)을 업데이트한다"는 관점은 AI 학습 전체를 관통합니다.

다음 노트북 `ch05_random_variables.ipynb`: 결과에 숫자를 붙이는 확률변수와 기댓값.